# Khảo sát dataset SR-BH 2020

Notebook này khảo sát trực tiếp file dữ liệu thật `data_capec_multilabel.csv` (không dựa vào mô tả trên website) để trả lời 7 câu hỏi khảo sát bắt buộc.

**Lưu ý quan trọng phát hiện được:** đây là dataset **multi-label** (mỗi request có thể mang nhiều nhãn CAPEC cùng lúc), không phải single-label như nhiều dataset tấn công web khác. Điều này ảnh hưởng đến cách trả lời câu 2 và câu 3.

In [17]:
import pandas as pd
pd.set_option('display.max_colwidth', 300)

from pathlib import Path

REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "preprocessing").is_dir() and (p / "reports").is_dir()
)
DATA_PATH = REPO_ROOT / "src" / "data" / "raw" / "data_capec_multilabel.csv"


df = pd.read_csv(DATA_PATH, low_memory=False)
print("Shape (rows, cols):", df.shape)
df.head(3)

Shape (rows, cols): (907815, 38)


,timestamp,src_ip,src_port,dst_ip,dst_port,request_http_method,request_http_request,request_http_protocol,request_user_agent,request_referer,...,126 - Path Traversal,66 - SQL Injection,16 - Dictionary-based Password Attack,310 - Scanning for Vulnerable Software,153 - Input Data Manipulation,248 - Command Injection,274 - HTTP Verb Tampering,194 - Fake the Source of Data,34 - HTTP Response Splitting,33 - HTTP Request Smuggling
0,17/Jul/2020:12:23:34 +0100,172.26.0.1,55894.0,172.26.0.4,80.0,GET,/,HTTP/1.1,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/44.0.2403.130 Safari/537.36",NaN,...,0,0,0,0,0,0,0,0,0,0
1,17/Jul/2020:12:23:41 +0100,172.26.0.1,55897.0,172.26.0.4,80.0,GET,/blog/index.php/2020/04/04/voluptatum-reprehenderit-maiores-ab-sequi-quaerat/,HTTP/1.1,Mozilla/5.0 (Macintosh; Intel Mac OS X 10.6; rv:36.0) Gecko/20100101 Firefox/36.0,NaN,...,0,0,0,0,0,0,0,0,0,0
2,17/Jul/2020:12:23:51 +0100,172.26.0.1,55901.0,172.26.0.4,80.0,GET,/blog/xmlrpc.php?rsd,HTTP/1.1,Mozilla/4.0 (compatible; MSIE 6.0; Windows NT 5.1; WOW64; Trident/4.0; SLCC2; .NET CLR 2.0.50727; .NET CLR 3.5.30729; .NET CLR 3.0.30729; .NET4.0C; .NET4.0E),NaN,...,0,0,0,0,0,0,0,0,0,0


## Câu 1: Tổng số bản ghi (request)

Đếm số dòng dữ liệu thực tế (không tính header).

In [18]:
total_records = len(df)
print(f"Tổng số bản ghi: {total_records:,}")

Tổng số bản ghi: 907,815


## Câu 2: Số nhãn/lớp tấn công và phân bố

Dataset gán nhãn theo chuẩn **CAPEC** (Common Attack Pattern Enumeration and Classification), mỗi loại tấn công là **một cột nhị phân riêng** (one-hot per label, không phải 1 cột "label" duy nhất) vì một request có thể vừa là SQLi vừa là Fake-Source-of-Data cùng lúc (multi-label).

=> "Số lớp" ở đây = số cột nhãn = 14 lớp (1 lớp `000 - Normal` + 13 lớp tấn công).

In [19]:
label_cols = [c for c in df.columns if c.split(' - ')[0].strip().isdigit()]
print(f"Số cột nhãn: {len(label_cols)}")
label_cols

Số cột nhãn: 14


['000 - Normal',
 '272 - Protocol Manipulation',
 '242 - Code Injection',
 '88 - OS Command Injection',
 '126 - Path Traversal',
 '66 - SQL Injection',
 '16 - Dictionary-based Password Attack',
 '310 - Scanning for Vulnerable Software',
 '153 - Input Data Manipulation',
 '248 - Command Injection',
 '274 - HTTP Verb Tampering',
 '194 - Fake the Source of Data',
 '34 - HTTP Response Splitting',
 '33 - HTTP Request Smuggling']

In [20]:
# Bảng phân bố số lượng mẫu mỗi lớp (số request có nhãn đó = 1)
class_dist = df[label_cols].sum().sort_values(ascending=False).astype(int)
class_dist_df = class_dist.reset_index()
class_dist_df.columns = ['Nhãn (CAPEC)', 'Số lượng mẫu']
class_dist_df['Tỷ lệ (%)'] = (class_dist_df['Số lượng mẫu'] / total_records * 100).round(3)
class_dist_df

,Nhãn (CAPEC),Số lượng mẫu,Tỷ lệ (%)
0,000 - Normal,525195,57.853
1,66 - SQL Injection,250311,27.573
2,194 - Fake the Source of Data,56145,6.185
3,126 - Path Traversal,20992,2.312
4,34 - HTTP Response Splitting,19738,2.174
5,242 - Code Injection,15827,1.743
6,272 - Protocol Manipulation,9153,1.008
7,88 - OS Command Injection,7482,0.824
8,274 - HTTP Verb Tampering,5437,0.599
9,310 - Scanning for Vulnerable Software,2718,0.299


In [21]:
# Kiểm tra tính multi-label: 1 request có thể có bao nhiêu nhãn cùng lúc?
n_labels_per_row = df[label_cols].sum(axis=1)
print("Phân bố số nhãn/1 request:")
print(n_labels_per_row.value_counts().sort_index())
print()
print("Số request KHÔNG có nhãn nào (lỗi gán nhãn nếu có):", (n_labels_per_row == 0).sum())

Phân bố số nhãn/1 request:
1    898577
2      8132
3      1088
4        18
Name: count, dtype: int64

Số request KHÔNG có nhãn nào (lỗi gán nhãn nếu có): 0


## Câu 3: Mức độ mất cân bằng dữ liệu

So sánh lớp lớn nhất và lớp nhỏ nhất (khác 0) trong 14 lớp.

In [22]:
max_class = class_dist.idxmax()
max_count = class_dist.max()
nonzero = class_dist[class_dist > 0]
min_class = nonzero.idxmin()
min_count = nonzero.min()

print(f"Lớp lớn nhất : {max_class}  -> {max_count:,} mẫu")
print(f"Lớp nhỏ nhất : {min_class}  -> {min_count:,} mẫu")
print(f"Tỷ lệ mất cân bằng (max/min) = {max_count/min_count:,.0f} lần")
print()
print("=> Kết luận: Dữ liệu mất cân bằng CỰC KỲ NGHIÊM TRỌNG.")
print("   - Lớp '000 - Normal' và '66 - SQL Injection' chiếm áp đảo.")
print(f"   - Lớp '{min_class}' chỉ có {min_count} mẫu duy nhất trên tổng {total_records:,} request.")

Lớp lớn nhất : 000 - Normal  -> 525,195 mẫu
Lớp nhỏ nhất : 248 - Command Injection  -> 1 mẫu
Tỷ lệ mất cân bằng (max/min) = 525,195 lần

=> Kết luận: Dữ liệu mất cân bằng CỰC KỲ NGHIÊM TRỌNG.
   - Lớp '000 - Normal' và '66 - SQL Injection' chiếm áp đảo.
   - Lớp '248 - Command Injection' chỉ có 1 mẫu duy nhất trên tổng 907,815 request.


## Câu 4: Cấu trúc mỗi bản ghi (schema)

Liệt kê toàn bộ trường dữ liệu và kiểu dữ liệu tương ứng.

In [23]:
schema = pd.DataFrame({
    'Trường': df.columns,
    'Kiểu dữ liệu': df.dtypes.astype(str).values,
    'Số giá trị null': df.isna().sum().values,
    '% null': (df.isna().sum().values / total_records * 100).round(2)
})
schema

,Trường,Kiểu dữ liệu,Số giá trị null,% null
0,timestamp,str,2,0.00
1,src_ip,str,2,0.00
2,src_port,float64,2,0.00
3,dst_ip,str,2,0.00
4,dst_port,float64,2,0.00
5,request_http_method,str,2,0.00
6,request_http_request,str,2,0.00
7,request_http_protocol,str,2,0.00
8,request_user_agent,str,3834,0.42
9,request_referer,str,616421,67.90


In [24]:
non_label_cols = [c for c in df.columns if c not in label_cols]
print("Các trường mô tả 1 HTTP request/response (không tính cột nhãn):")
for c in non_label_cols:
    print(" -", c)

Các trường mô tả 1 HTTP request/response (không tính cột nhãn):
 - timestamp
 - src_ip
 - src_port
 - dst_ip
 - dst_port
 - request_http_method
 - request_http_request
 - request_http_protocol
 - request_user_agent
 - request_referer
 - request_host
 - request_origin
 - request_cookie
 - request_content_type
 - request_accept
 - request_accept_language
 - request_accept_encoding
 - request_do_not_track
 - request_connection
 - request_body
 - response_http_protocol
 - response_http_status_code
 - response_http_status_message
 - response_content_length


## Câu 5: Ví dụ cụ thể (dán nguyên văn payload)

Trích 3 ví dụ thật từ dataset: 1 request lành tính (Normal), 1 request SQL Injection có cú pháp SQL rõ ràng, 1 request XSS.

In [25]:
print("=" * 90)
print("VÍ DỤ 1 - REQUEST LÀNH TÍNH (Normal)")
print("=" * 90)
normal_row = df[df['000 - Normal'] == 1].iloc[0]
print("Method  :", normal_row['request_http_method'])
print("URL     :", normal_row['request_http_request'])
print("Body    :", normal_row['request_body'])
print("User-Agent:", normal_row['request_user_agent'])

VÍ DỤ 1 - REQUEST LÀNH TÍNH (Normal)
Method  : GET
URL     : /
Body    : nan
User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/44.0.2403.130 Safari/537.36


In [26]:
print("=" * 90)
print("VÍ DỤ 2 - SQL INJECTION (payload SQL rõ ràng trong URL)")
print("=" * 90)
sqli_mask = df['66 - SQL Injection'] == 1
sql_pattern = r"union\s+select|information_schema|%27|--|sleep\(|benchmark\("
sqli_visible = df[sqli_mask & (
    df['request_http_request'].astype(str).str.contains(sql_pattern, case=False, regex=True, na=False)
)]
sqli_row = sqli_visible.iloc[0]
print("Method  :", sqli_row['request_http_method'])
print("URL     :", sqli_row['request_http_request'])
print("Body    :", sqli_row['request_body'])
print()
print("(Payload giải mã URL-decode để dễ đọc:)")
import urllib.parse
print(urllib.parse.unquote(str(sqli_row['request_http_request'])))

VÍ DỤ 2 - SQL INJECTION (payload SQL rõ ràng trong URL)
Method  : GET
URL     : /?p4yl04d2=1%20UNION%20ALL%20SELECT%201,2,3,table_name%20FROM%20information_schema.tables
Body    : nan

(Payload giải mã URL-decode để dễ đọc:)
/?p4yl04d2=1 UNION ALL SELECT 1,2,3,table_name FROM information_schema.tables


In [27]:
print("=" * 90)
print("VÍ DỤ 3 - XSS (Cross-Site Scripting)")
print("=" * 90)
xss_mask = df['request_http_request'].astype(str).str.contains(
    r'<script|onerror=|alert\(|javascript:|onload=', case=False, regex=True, na=False
)
print(f"Số request chứa payload dạng XSS rõ ràng trong URL: {xss_mask.sum()}")
xss_row = df[xss_mask].iloc[0]
print("Method  :", xss_row['request_http_method'])
print("URL     :", xss_row['request_http_request'])
print("Body    :", xss_row['request_body'])
print()
print("Nhãn CAPEC được gán cho request XSS này:")
print(xss_row[label_cols][xss_row[label_cols] == 1])
print()
print(">> LƯU Ý: dataset KHÔNG có nhãn 'XSS' riêng biệt.")
print(">> XSS trong dataset này được gộp vào nhãn '242 - Code Injection'.")

VÍ DỤ 3 - XSS (Cross-Site Scripting)
Số request chứa payload dạng XSS rõ ràng trong URL: 1
Method  : GET
URL     : /?p4yl04d3=<script>alert(document.cookie)</script>
Body    : nan

Nhãn CAPEC được gán cho request XSS này:
242 - Code Injection                      1
310 - Scanning for Vulnerable Software    1
Name: 7195, dtype: object

>> LƯU Ý: dataset KHÔNG có nhãn 'XSS' riêng biệt.
>> XSS trong dataset này được gộp vào nhãn '242 - Code Injection'.


## Câu 6: Dataset đã chia sẵn train/test chưa?

Kiểm tra xem có cột đánh dấu tập (split/set/train/test) hoặc có nhiều file (train.csv, test.csv...) hay không.

In [28]:
import os

split_like_cols = [c for c in df.columns if any(k in c.lower() for k in ['split', 'set', 'train', 'test', 'fold'])]
print("Cột liên quan đến train/test tìm thấy:", split_like_cols if split_like_cols else "KHÔNG CÓ")

data_dir = os.path.dirname(DATA_PATH)
print()
print("Danh sách file trong thư mục dataset:")
for f in os.listdir(data_dir):
    print(" -", f)

Cột liên quan đến train/test tìm thấy: ['34 - HTTP Response Splitting']

Danh sách file trong thư mục dataset:
 - data_capec_multilabel.csv
 - README.md


**Kết luận câu 6:** Dataset chỉ gồm **1 file CSV duy nhất** (`data_capec_multilabel.csv`), không có cột split và không có sẵn file train/test riêng biệt.

**Đề xuất chia tập:** Vì dữ liệu có timestamp tuần tự (log truy cập theo thời gian) và mất cân bằng nghiêm trọng (câu 3), nên:
- Chia theo **tỷ lệ 70/15/15** (train/val/test) hoặc **80/20** (train/test).
- Dùng **stratified split** theo nhãn chính (ít nhất theo `000 - Normal` vs attack, lý tưởng là stratify đa nhãn qua `iterative-stratification`) để đảm bảo các lớp hiếm (vd: `248 - Command Injection` chỉ có 1 mẫu) không bị dồn hết vào 1 tập.
- Cân nhắc chia theo **thời gian** (time-based split: dữ liệu sớm -> train, dữ liệu muộn -> test) thay vì random, vì đây là dữ liệu log tuần tự, tránh rò rỉ thông tin (data leakage) giữa các request cùng phiên.

## Câu 7 (then chốt): Cặp ví dụ benign vs attack rất giống nhau

**Phương pháp:** Tìm các request tấn công có payload "ẩn" bên trong tham số của form hợp lệ (thay vì URL lộ liễu như câu 5). Cụ thể: lọc tất cả request POST đến cùng 1 endpoint (`/blog/index.php/my-account/edit-password/`), rồi so khớp độ tương đồng chuỗi (difflib) giữa các request Normal và các request được gán nhãn SQL Injection.

In [29]:
import difflib

target_path = '/blog/index.php/my-account/edit-password/'
same_endpoint = df[(df['request_http_request'] == target_path) & (df['request_http_method'] == 'POST')]

print(f"Tổng số request POST tới '{target_path}': {len(same_endpoint)}")
print(same_endpoint[label_cols].sum()[same_endpoint[label_cols].sum() > 0])

Tổng số request POST tới '/blog/index.php/my-account/edit-password/': 2033
000 - Normal                       4
272 - Protocol Manipulation       34
242 - Code Injection             131
88 - OS Command Injection        488
126 - Path Traversal             290
66 - SQL Injection               807
153 - Input Data Manipulation    264
194 - Fake the Source of Data    363
34 - HTTP Response Splitting      84
dtype: int64


In [30]:
normal_bodies = same_endpoint[same_endpoint['000 - Normal'] == 1]['request_body'].dropna().unique()
sqli_bodies = same_endpoint[same_endpoint['66 - SQL Injection'] == 1]['request_body'].dropna().unique()

best = None
for nb in normal_bodies:
    for sb in sqli_bodies:
        ratio = difflib.SequenceMatcher(None, nb, sb).ratio()
        if best is None or ratio > best[0]:
            best = (ratio, nb, sb)

print(f"Độ tương đồng cao nhất tìm được: {best[0]*100:.1f}%")
print()
print("--- Request LÀNH TÍNH (Normal) ---")
print(best[1])
print()
print("--- Request TẤN CÔNG (SQL Injection) ---")
print(best[2])

Độ tương đồng cao nhất tìm được: 76.2%

--- Request LÀNH TÍNH (Normal) ---
password_current=HShsaZxy&password_1=HShsaZxy&password_2=HShsaZxy&_wpnonce=b9dad6a8b5&_wp_http_referer=&save_change_password=Save+changes&action=save_change_password

--- Request TẤN CÔNG (SQL Injection) ---
password_current=HBY30D-&password_1=HBY30D-&password_2=HBY30D-&_wpnonce=b9dad6a8b5&_wp_http_referer=%2Fblog%2Findex.php%2Fmy-account%2Fedit-password%2F&save_change_password=Save+changes&action=save_change_password


In [31]:
normal_full = same_endpoint[(same_endpoint['request_body'] == best[1]) & (same_endpoint['000 - Normal'] == 1)].iloc[0]
attack_full = same_endpoint[(same_endpoint['request_body'] == best[2]) & (same_endpoint['66 - SQL Injection'] == 1)].iloc[0]

cols_show = ['timestamp', 'src_ip', 'request_http_method', 'request_http_request', 'request_body', 'response_http_status_code']

print("=" * 100)
print("REQUEST A - GÁN NHÃN: LÀNH TÍNH (000 - Normal = 1)")
print("=" * 100)
for c in cols_show:
    print(f"{c:28s}: {normal_full[c]}")

print()
print("=" * 100)
print("REQUEST B - GÁN NHÃN: TẤN CÔNG (66 - SQL Injection = 1)")
print("=" * 100)
for c in cols_show:
    print(f"{c:28s}: {attack_full[c]}")

REQUEST A - GÁN NHÃN: LÀNH TÍNH (000 - Normal = 1)
timestamp                   : 18/Jul/2020:09:00:35 +0200
src_ip                      : 172.26.0.1
request_http_method         : POST
request_http_request        : /blog/index.php/my-account/edit-password/
request_body                : password_current=HShsaZxy&password_1=HShsaZxy&password_2=HShsaZxy&_wpnonce=b9dad6a8b5&_wp_http_referer=&save_change_password=Save+changes&action=save_change_password
response_http_status_code   : 200.0

REQUEST B - GÁN NHÃN: TẤN CÔNG (66 - SQL Injection = 1)
timestamp                   : 18/Jul/2020:00:35:18 +0200
src_ip                      : 172.26.0.1
request_http_method         : POST
request_http_request        : /blog/index.php/my-account/edit-password/
request_body                : password_current=HBY30D-&password_1=HBY30D-&password_2=HBY30D-&_wpnonce=b9dad6a8b5&_wp_http_referer=%2Fblog%2Findex.php%2Fmy-account%2Fedit-password%2F&save_change_password=Save+changes&action=save_change_password
respon

### Nhận xét (Figure minh họa động lực nghiên cứu)

Hai request trên gần như **giống hệt nhau về cấu trúc bề mặt**:
- Cùng endpoint, cùng HTTP method (POST), cùng bộ tham số form (`password_current`, `password_1`, `password_2`, `_wpnonce`, ...).
- Cùng nguồn IP giả lập, cùng HTTP status trả về (200 OK).
- Giá trị mật khẩu ở cả hai chỉ là chuỗi ký tự ngẫu nhiên (`HShsaZxy` vs `HBY30D-`) — **không hề chứa cú pháp SQL** (không có `' OR 1=1`, `UNION SELECT`, `--`, ...) trong request được gán nhãn SQL Injection.

=> Đây chính là **bằng chứng trực quan cho gap của đề tài**: một bộ phân loại/WAF chỉ dựa vào **nội dung payload bề mặt (content/signature-based)** sẽ **không thể phân biệt** được request A và B, vì khác biệt không nằm ở cú pháp tấn công lộ liễu mà (nhiều khả năng) nằm ở **ngữ cảnh hành vi/phiên làm việc** (request này thuộc một chuỗi tấn công dò mật khẩu/SQLi tự động ở cấp độ session, không thể hiện qua 1 request đơn lẻ).

=> Động lực nghiên cứu: cần phương pháp phát hiện tấn công dựa trên **ngữ cảnh/chuỗi hành vi** (sequence/behavior-based), không chỉ dựa vào pattern-matching nội dung của từng request riêng lẻ.

## Tổng kết nhanh

| Câu hỏi | Trả lời ngắn gọn |
|---|---|
| 1. Tổng số bản ghi | 907,815 request |
| 2. Số nhãn | 14 nhãn CAPEC (1 Normal + 13 loại tấn công), multi-label |
| 3. Mất cân bằng | Rất nghiêm trọng — lớp lớn nhất gấp ~525,195 lần lớp nhỏ nhất (`248 - Command Injection` chỉ có 1 mẫu) |
| 4. Số trường | 24 trường HTTP request/response + 14 cột nhãn = 38 cột |
| 5. Ví dụ | Xem Câu 5 — payload SQLi/XSS trích nguyên văn |
| 6. Train/test | Chưa chia sẵn — đề xuất 70/15/15 hoặc 80/20, stratified/time-based split |
| 7. Cặp benign/attack giống nhau | Tìm được — cùng endpoint `edit-password`, thân request gần như giống hệt (~76% tương đồng), khác biệt không nằm ở payload mà ở ngữ cảnh |


## Q7 - Phân tích kỹ thuật: Nguyên nhân 2 request giống nhau nhưng nhãn khác nhau (Research Gap / Motivating Example)

### 1. Phân tích chi tiết bề mặt Payload (Surface Payload Analysis)

So sánh trực diện 2 request POST tới endpoint `/blog/index.php/my-account/edit-password/`:

* **Request A (Gán nhãn: `000 - Normal`)**:
  * Body: `password_current=HShsaZxy&password_1=HShsaZxy&password_2=HShsaZxy&_wpnonce=b9dad6a8b5&_wp_http_referer=&save_change_password=Save+changes&action=save_change_password`
* **Request B (Gán nhãn: `66 - SQL Injection`)**:
  * Body: `password_current=HBY30D-&password_1=HBY30D-&password_2=HBY30D-&_wpnonce=b9dad6a8b5&_wp_http_referer=%2Fblog%2Findex.php%2Fmy-account%2Fedit-password%2F&save_change_password=Save+changes&action=save_change_password`

**Đặc điểm tương đồng (>76% similarity):**
1. Cùng HTTP Method (`POST`) và cùng Endpoint target (`/blog/index.php/my-account/edit-password/`).
2. Cùng cấu trúc Form data gồm 7 tham số chuẩn của WordPress/WooCommerce.
3. Chuỗi giá trị mật khẩu trong Request B (`HBY30D-`) **hoàn toàn KHÔNG chứa cú pháp SQL Injection điển hình** (không chứa `' OR '1'='1`, `UNION SELECT`, `SLEEP()`, `--`, `#`, v.v.).
4. Cùng trả về HTTP Status Code `200 OK` từ server.

---

### 2. Giải mã nguyên nhân kỹ thuật (Why Labels Differ)

Vì sao Request B không có cú pháp SQL nhưng lại bị gán nhãn `66 - SQL Injection`?

1. **Phương pháp gán nhãn theo phiên/chiến dịch (Session/Campaign-based Labeling)**:
   * Trong quá trình thu thập dữ liệu sinh ra dataset SR-BH 2020, kẻ tấn công (hoặc script/tool tự động như `sqlmap`, `Wfuzz`, công cụ fuzzing) đã thực hiện một **chiến dịch dò quét lỗ hổng SQLi** trên ứng dụng web.
   * Script tự động gửi hàng loạt request liên tiếp để thăm dò phản hồi, bao gồm cả các bước khởi tạo phiên, thử nghiệm giá trị biên (như thêm dấu `-` vào chuỗi `HBY30D-`), hoặc lấy giá trị `_wpnonce`.
   * Đơn vị thu thập dataset đã gán nhãn dựa trên **toàn bộ lưu lượng truy cập phát ra từ tiến trình/IP của công cụ tấn công** trong khoảng thời gian diễn ra chiến dịch. Vì vậy, các request phụ trợ hoặc request fuzzing sơ khai nằm trong chuỗi tấn công này đều bị gán nhãn `66 - SQL Injection`.

2. **Dấu hiệu nhỏ về hành vi tự động**:
   * `_wp_http_referer`: Request B có giá trị referer được URL-encode sẵn `%2Fblog%2Findex.php%2Fmy-account%2Fedit-password%2F`, cho thấy request này được tạo tự động bởi script/bot thay vì thao tác trình duyệt tự nhiên (Request A có referer rỗng).
   * Ký tự `-` cuối chuỗi `HBY30D-`: Thường được công cụ fuzzing chèn vào để kiểm tra phản hồi xử lý ký tự đặc biệt của cơ sở dữ liệu backend.

---

### 3. Động lực nghiên cứu & Khoảng trống kiến thức (Research Gap & Motivation)

Ví dụ thực nghiệm trên chỉ ra một **khoảng trống rất lớn (Research Gap)** trong các phương pháp phát hiện tấn công web hiện nay:

* **Thất bại của mô hình Content-Based / Single-Request Classification**:
  * Các hệ thống WAF truyền thống (dùng RegEx/Signature) hoặc các mô hình Machine Learning / Deep Learning chỉ phân tích đơn lẻ từng request ($P(y \mid r_t)$) dựa trên đặc trưng chuỗi (N-gram, TF-IDF, Word2Vec) sẽ **chắc chắn phân loại sai Request B thành Normal** (bị lỗi False Negative).
  * Lý do: Bản thân nội dung Request B không chứa bất kỳ pattern độc hại nào để mô hình content-based có thể nhận biết.

* **Yêu cầu bắt buộc chuyển sang Sequence-based & Context-aware Detection**:
  * Để phân loại đúng các request như Request B, mô hình bắt buộc phải học ngữ cảnh chuỗi thời gian ($P(y_t \mid r_t, r_{t-1}, \dots, r_{t-k})$) và thông tin phiên (Session/IP context).
  * Việc nhận diện tấn công không chỉ phụ thuộc vào *Request đó chứa gì* (Content), mà phải phụ thuộc vào *Request đó nằm trong chuỗi hành vi nào* (Behavior Sequence & Timing Context).
